<a href="https://colab.research.google.com/github/nwilliamsanalytics/content-investment-roi/blob/main/Copy_of_Streamer_Subscriber_Strength_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Subscriber Stength Analysis**



1.   Do larger platforms actually retain users better?
2.   Which platform has the highest subscriber lifetime value?

1.   Is growth being driven by retention or replacement?
2.   Are high-subscriber platforms masking weak retention?






In [ ]:
#IMPORT package

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#IMPORT platform subscriber data

from google.colab import files
uploaded = files.upload()

In [ ]:
#LOAD file

df = pd.read_csv('SVOD_Platform_Count.csv')
df.head()

,Platform,Subscribers (Estimate)
0,Netflix,"325,000,000"
1,Amazon Prime,"200,000,000"
2,Disney+,"131,600,000"
3,"HBO Maxincluding HBO, Discovery+ and other WBD","131,600,000"
4,Tencent Video,"114,000,000"


In [ ]:
#CLEAN dataset


# Rename column
df = df.rename(columns={"Subscribers (Estimate)": "Subscribers"})

# Remove commas and convert to numeric
df["Subscribers"] = df["Subscribers"].replace(",", "", regex=True).astype(float)

# Clean platform names
df["Platform"] = df["Platform"].replace({
    "Amazon Prime": "Amazon Prime Video",
    "Apple TV": "Apple TV+",
    "HBO Maxincluding HBO, Discovery+ and other WBD": "HBO Max"
})

# View cleaned data
df.head()

,Platform,Subscribers
0,Netflix,325000000.0
1,Amazon Prime Video,200000000.0
2,Disney+,131600000.0
3,HBO Max,131600000.0
4,Tencent Video,114000000.0


In [ ]:
#BUILD ARPU - Average revenue per user - data

# Add ARPU (monthly, in USD)
arpu_values = {
    "Netflix": 15,
    "Amazon Prime Video": 12,
    "Disney+": 10,
    "HBO Max": 14,
    "Tencent Video": 8,
    "iQIYI": 7,
    "JioHotstar": 5,
    "Paramount+": 9,
    "Hulu": 13,
    "Peacock": 8,
    "Hotstar": 5,
    "SonyLiv": 6,
    "Apple TV+": 11,
    "iFlix": 4,
    "CuriosityStream": 3,
    "ESPN": 12
}

df["ARPU"] = df["Platform"].map(arpu_values)

df.head()

,Platform,Subscribers,ARPU
0,Netflix,325000000.0,15.0
1,Amazon Prime Video,200000000.0,12.0
2,Disney+,131600000.0,10.0
3,HBO Max,131600000.0,14.0
4,Tencent Video,114000000.0,8.0


In [ ]:
#BUILD annual revenue data

# Calculate annual revenue
df["Annual_Revenue"] = df["Subscribers"] * df["ARPU"] * 12

# Format for readability (optional but helpful)
df["Annual_Revenue_Billions"] = df["Annual_Revenue"] / 1e9

df.head()

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions
0,Netflix,325000000.0,15.0,5.850000e+10,58.5000
1,Amazon Prime Video,200000000.0,12.0,2.880000e+10,28.8000
2,Disney+,131600000.0,10.0,1.579200e+10,15.7920
3,HBO Max,131600000.0,14.0,2.210880e+10,22.1088
4,Tencent Video,114000000.0,8.0,1.094400e+10,10.9440


In [ ]:
#CHECK if ARPU was asigned to all platforms in dataset

df[df["ARPU"].isna()]

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions
16,StarTimes,20000000.0,NaN,NaN,NaN
17,DAZN,20000000.0,NaN,NaN,NaN
18,Crunchyroll,17000000.0,NaN,NaN,NaN
19,Viu,13800000.0,NaN,NaN,NaN
20,Starz,12200000.0,NaN,NaN,NaN
...,...,...,...,...,...
86,Blim TV,297000.0,NaN,NaN,NaN
87,Sweet TV,150000.0,NaN,NaN,NaN
88,Foxtel,131000.0,NaN,NaN,NaN
89,Volia TV,120000.0,NaN,NaN,NaN


In [ ]:
#CREATE platform tiers to assign ARPU for all platforms

def assign_arpu(platform):
    platform = platform.lower()

    # Premium global platforms
    if any(x in platform for x in ["netflix", "hbo", "hulu", "disney", "prime", "apple", "paramount"]):
        return 12

    # Mid-tier / mixed monetization
    elif any(x in platform for x in ["peacock", "espn", "sony", "viu", "wetv"]):
        return 8

    # Regional / emerging markets
    elif any(x in platform for x in ["iqiyi", "tencent", "hotstar", "jio", "iflix"]):
        return 5

    # Niche / smaller platforms
    else:
        return 6

In [ ]:
#APPLY tiers to data

df["ARPU"] = df["Platform"].apply(assign_arpu)
df.head()

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions
0,Netflix,325000000.0,12,5.850000e+10,58.5000
1,Amazon Prime Video,200000000.0,12,2.880000e+10,28.8000
2,Disney+,131600000.0,12,1.579200e+10,15.7920
3,HBO Max,131600000.0,12,2.210880e+10,22.1088
4,Tencent Video,114000000.0,5,1.094400e+10,10.9440


In [ ]:
#CHECK for NaN values

df["ARPU"].isna().sum()

np.int64(0)

In [ ]:
#RECALCULATE revenue

# Recalculate annual revenue
df["Annual_Revenue"] = df["Subscribers"] * df["ARPU"] * 12

# Convert to billions for readability
df["Revenue_Billions"] = df["Annual_Revenue"] / 1e9

df.head()

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions,Revenue_Billions
0,Netflix,325000000.0,12,4.680000e+10,58.5000,46.8000
1,Amazon Prime Video,200000000.0,12,2.880000e+10,28.8000,28.8000
2,Disney+,131600000.0,12,1.895040e+10,15.7920,18.9504
3,HBO Max,131600000.0,12,1.895040e+10,22.1088,18.9504
4,Tencent Video,114000000.0,5,6.840000e+09,10.9440,6.8400


In [ ]:
df.columns

Index(['Platform', 'Subscribers', 'ARPU', 'Annual_Revenue',
       'Annual_Revenue_Billions', 'Revenue_Billions'],
      dtype='object')

In [ ]:
#CREATE Churn

def refine_churn(platform):
    platform = platform.lower()

    if "netflix" in platform:
        return 0.025
    elif "disney" in platform:
        return 0.035
    elif "hbo" in platform:
        return 0.033
    elif "prime" in platform:
        return 0.03
    elif "apple" in platform:
        return 0.04
    elif "hulu" in platform:
        return 0.032
    elif "paramount" in platform:
        return 0.045
    elif any(x in platform for x in ["iqiyi", "tencent", "hotstar", "jio", "iflix"]):
        return 0.06
    else:
        return 0.05

df["Churn_Rate"] = df["Platform"].apply(refine_churn)

In [ ]:
#CALCULATE churn - Lifetime of subscriber, revenue over lifetime
#ADD column - Lifetime Months - mhow long subscribers stay on the platform
#ADD column - LTV - Lifetime value of subscribers

# Calculate subscriber lifetime in months
df["Lifetime_Months"] = 1 / df["Churn_Rate"]

# Calculate lifetime value
df["LTV"] = df["ARPU"] * df["Lifetime_Months"]

df.head()

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions,Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,12,4.680000e+10,58.5000,46.8000,0.025,40.000000,480.000000
1,Amazon Prime Video,200000000.0,12,2.880000e+10,28.8000,28.8000,0.030,33.333333,400.000000
2,Disney+,131600000.0,12,1.895040e+10,15.7920,18.9504,0.035,28.571429,342.857143
3,HBO Max,131600000.0,12,1.895040e+10,22.1088,18.9504,0.033,30.303030,363.636364
4,Tencent Video,114000000.0,5,6.840000e+09,10.9440,6.8400,0.060,16.666667,83.333333


In [ ]:
#REMOVE duplicate column


df = df.drop(columns=["Annual_Revenue"])

In [ ]:
df.columns

Index(['Platform', 'Subscribers', 'ARPU', 'Revenue_Billions', 'Churn_Rate',
       'Lifetime_Months', 'LTV'],
      dtype='object')

In [ ]:
df.head()

,Platform,Subscribers,ARPU,Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,12,46.8000,0.025,40.000000,480.000000
1,Amazon Prime Video,200000000.0,12,28.8000,0.030,33.333333,400.000000
2,Disney+,131600000.0,12,18.9504,0.035,28.571429,342.857143
3,HBO Max,131600000.0,12,18.9504,0.033,30.303030,363.636364
4,Tencent Video,114000000.0,5,6.8400,0.060,16.666667,83.333333


In [ ]:
#RECALCULATE

# Recalculate lifetime and LTV
df["Lifetime_Months"] = 1 / df["Churn_Rate"]
df["LTV"] = df["ARPU"] * df["Lifetime_Months"]

df.head()

,Platform,Subscribers,ARPU,Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,12,46.8000,0.025,40.000000,480.000000
1,Amazon Prime Video,200000000.0,12,28.8000,0.030,33.333333,400.000000
2,Disney+,131600000.0,12,18.9504,0.035,28.571429,342.857143
3,HBO Max,131600000.0,12,18.9504,0.033,30.303030,363.636364
4,Tencent Video,114000000.0,5,6.8400,0.060,16.666667,83.333333


In [ ]:

# Sort by LTV
df.sort_values(by="LTV", ascending=False).head(10)

,Platform,Subscribers,ARPU,Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,12,46.8000,0.025,40.000000,480.000000
1,Amazon Prime Video,200000000.0,12,28.8000,0.030,33.333333,400.000000
8,Hulu,64100000.0,12,9.2304,0.032,31.250000,375.000000
44,Hulu (Nippon),3000000.0,12,0.4320,0.032,31.250000,375.000000
3,HBO Max,131600000.0,12,18.9504,0.033,30.303030,363.636364
2,Disney+,131600000.0,12,18.9504,0.035,28.571429,342.857143
12,Apple TV+,30000000.0,12,4.3200,0.040,25.000000,300.000000
7,Paramount+,78900000.0,12,11.3616,0.045,22.222222,266.666667
19,Viu,13800000.0,8,1.3248,0.050,20.000000,160.000000
11,SonyLiv,33300000.0,8,3.1968,0.050,20.000000,160.000000


Hypothesis:

Subscriber scale alone does not determine business strength—retention (churn) materially impacts lifetime value, and smaller platforms with stronger retention can outperform larger platforms on a per-user basis.

In [ ]:
#EXPORT data to Tableau for visualization

df.to_csv("svod_model_output.csv", index=False)

In [ ]:
z#EXPORT data for Tableau viz

df.to_csv("streaming_ltv_analysis.csv", index=False)

In [ ]:
#CREATE more variation in the data

def refine_churn(subscribers, arpu):

    # Mega platforms (very stable)
    if subscribers >= 250_000_000:
        return 0.022

    # Large platforms
    elif subscribers >= 150_000_000:
        return 0.025

    # Upper mid-tier
    elif subscribers >= 100_000_000:
        return 0.030

    # Mid-tier
    elif subscribers >= 50_000_000:
        return 0.035

    # Smaller platforms
    elif subscribers >= 20_000_000:
        return 0.045

    # Niche platforms
    else:
        return 0.055 if arpu >= 10 else 0.065

In [ ]:
df["Churn_Rate"] = df.apply(lambda row: refine_churn(row["Subscribers"], row["ARPU"]), axis=1)

In [ ]:
#RECALCULATE

df["Lifetime_Months"] = 1 / df["Churn_Rate"]
df["LTV"] = df["ARPU"] * df["Lifetime_Months"]

In [ ]:
df.columns

Index(['Platform', 'Subscribers', 'ARPU', 'Revenue_Billions', 'Churn_Rate',
       'Lifetime_Months', 'LTV'],
      dtype='object')

In [ ]:
df.head()

,Platform,Subscribers,ARPU,Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,12,46.8000,0.022,45.454545,545.454545
1,Amazon Prime Video,200000000.0,12,28.8000,0.025,40.000000,480.000000
2,Disney+,131600000.0,12,18.9504,0.030,33.333333,400.000000
3,HBO Max,131600000.0,12,18.9504,0.030,33.333333,400.000000
4,Tencent Video,114000000.0,5,6.8400,0.030,33.333333,166.666667


In [ ]:
import numpy as np
df["Churn_Rate"] += np.random.uniform(-0.003, 0.003, len(df))

In [ ]:
df.head()

,Platform,Subscribers,ARPU,Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,12,46.8000,0.020206,45.454545,545.454545
1,Amazon Prime Video,200000000.0,12,28.8000,0.025884,40.000000,480.000000
2,Disney+,131600000.0,12,18.9504,0.028342,33.333333,400.000000
3,HBO Max,131600000.0,12,18.9504,0.031562,33.333333,400.000000
4,Tencent Video,114000000.0,5,6.8400,0.028478,33.333333,166.666667


In [ ]:
df.head(20)

,Platform,Subscribers,ARPU,Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,12,46.8000,0.020206,45.454545,545.454545
1,Amazon Prime Video,200000000.0,12,28.8000,0.025884,40.000000,480.000000
2,Disney+,131600000.0,12,18.9504,0.028342,33.333333,400.000000
3,HBO Max,131600000.0,12,18.9504,0.031562,33.333333,400.000000
4,Tencent Video,114000000.0,5,6.8400,0.028478,33.333333,166.666667
5,iQIYI,101100000.0,5,6.0660,0.027595,33.333333,166.666667
6,JioHotstar,100000000.0,5,6.0000,0.032530,33.333333,166.666667
7,Paramount+,78900000.0,12,11.3616,0.035218,28.571429,342.857143
8,Hulu,64100000.0,12,9.2304,0.032961,28.571429,342.857143
9,Peacock,44000000.0,8,4.2240,0.045997,22.222222,177.777778


In [ ]:
#EXPORT

df.to_csv("svod_model_output2.csv", index=False)